# TCL Phase 3 PopQA Runs

Runs inference v2 for one model on the Phase 3 PopQA sample (`data/phase3/popqa.csv`), as fixed in `docs/phase3/TCL-Phase3-Protocol.md` (Section 3). Same settings as Phase 1: answer-span confidence, hidden states at 25/50/100% depth, P(True), semantic entropy with 10 samples.

Run one session per model (Qwen, then Phi). Each session writes a zip to `/kaggle/working/` that I download and analyze locally.

## Settings

- Accelerator: **GPU T4 x2** (the NLI model for semantic entropy runs on the second GPU).
- Internet: **on** (downloads Qwen, Phi, and the NLI model from Hugging Face).
- `N_QUESTIONS`: 1500 runs batch 1. If the size rule needs more (fewer than 150 correct or 150 incorrect answers), set it to 3000 or 4500 and re-run: finished questions are skipped.

In [ ]:
# Pin the exact code and data version used for this run.
REPO = "https://github.com/awabmoha/truth-calibration-layer.git"
COMMIT = "51e17024749087cc31291198bb89dd2a3c61a03a"

# Choose one model per session.
MODEL_TAG = "qwen"   # "qwen" or "phi"
MODELS = {
    "qwen": "Qwen/Qwen2.5-0.5B-Instruct",
    "phi": "microsoft/Phi-3.5-mini-instruct",
}
N_QUESTIONS = 1500   # 1500 = batch 1; 3000 / 4500 only if the size rule needs it
print(MODEL_TAG, MODELS[MODEL_TAG], N_QUESTIONS)

In [ ]:
import os, subprocess
if not os.path.exists("truth-calibration-layer"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("truth-calibration-layer")
subprocess.run(["git", "checkout", COMMIT], check=True)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
os.chdir("tcl_experiments")

In [ ]:
!python scripts/check_runtime.py
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Run

Resumable: if the session stops, re-running this cell continues where it left off.

In [ ]:
OUT_ROOT = f"/kaggle/working/phase3_popqa/{MODEL_TAG}"
out_dir = f"{OUT_ROOT}/popqa"
cmd = [
    "python", "scripts/run_inference_v2.py",
    "--model", MODELS[MODEL_TAG],
    "--model-tag", MODEL_TAG,
    "--dataset", "popqa",
    "--questions", "data/phase3/popqa.csv",
    "--out-dir", out_dir,
    "--se-splits", "test",
    "--limit", str(N_QUESTIONS),
    "--resume",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Package Output

In [ ]:
import shutil
with open(f"{out_dir}/records.jsonl") as f:
    print("popqa", sum(1 for _ in f), "records")
zip_path = shutil.make_archive(f"/kaggle/working/phase3_popqa_{MODEL_TAG}", "zip", OUT_ROOT)
print(zip_path, os.path.getsize(zip_path) // 1_000_000, "MB")